# Applied workflow: from a dataset to a checked model

A training API is the middle of a machine-learning project, not the project itself. This lesson follows one small classification problem from a stated decision through data inspection, a baseline, a PyTorch training loop, validation choices, and error analysis. The dataset is synthetic so every example and every source of variation remains visible.

The fastai `Learner` is discussed at the end as a way to package this workflow. The PyTorch loop comes first because it makes the responsibilities explicit.

In [ ]:
import random
from dataclasses import dataclass
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Define the observation and the decision

Each input is a noisy 16×16 image containing either a vertical bar (`0`) or a horizontal bar (`1`). The bar can move and vary in thickness. The model is not asked to recover the drawing process; it is asked to predict the class of a new image.

The split must happen before any learned preprocessing or model selection. We use a stratified train/validation/test split so the class proportions are comparable.

In [ ]:
def make_bars(n, size=16, noise=0.22):
    images = np.zeros((n, 1, size, size), dtype=np.float32)
    labels = np.random.randint(0, 2, n)
    for i, label in enumerate(labels):
        width = np.random.randint(2, 5)
        offset = np.random.randint(2, size - width - 2)
        if label == 0: images[i, 0, :, offset:offset + width] = 1.0
        else: images[i, 0, offset:offset + width, :] = 1.0
    images += np.random.normal(0, noise, images.shape).astype(np.float32)
    return np.clip(images, 0, 1), labels

X, y = make_bars(2400)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=.20, stratify=y_train, random_state=SEED)
print({name: value.shape for name, value in [('train', X_train), ('valid', X_valid), ('test', X_test)]})
print(pd.Series(y_train).value_counts(normalize=True).rename('proportion'))

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, image, label in zip(axes.flat, X_train[:12], y_train[:12]):
    ax.imshow(image[0], cmap='gray', vmin=0, vmax=1); ax.set_title(f'class {label}'); ax.axis('off')
plt.suptitle('Representative training examples'); plt.tight_layout()

## 2. Establish a baseline and choose metrics

The majority classifier ignores every pixel. Its accuracy is the minimum useful reference. Because the classes are balanced here, accuracy and balanced accuracy are similar; reporting both makes that assumption visible rather than accidental.

In [ ]:
majority = np.bincount(y_train).argmax()
print('majority accuracy:', (y_test == majority).mean())
print('majority balanced accuracy:', balanced_accuracy_score(y_test, np.full_like(y_test, majority)))

## 3. Train with a transparent PyTorch loop

The model returns logits, not probabilities. `CrossEntropyLoss` combines `log_softmax` and the negative log-likelihood for the integer target. Validation runs with `eval()` and `no_grad()` so dropout or batch-normalization behavior cannot leak into the measurement.

In [ ]:
def loader(X, y, batch_size, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long)), batch_size=batch_size, shuffle=shuffle)
train_loader = loader(X_train, y_train, 64, True); valid_loader = loader(X_valid, y_valid, 256); test_loader = loader(X_test, y_test, 256)

model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                      nn.Flatten(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def evaluate(loader):
    model.eval(); total_loss=correct=count=0
    with torch.no_grad():
        for xb, yb in loader:
            logits=model(xb.to(device)); yb=yb.to(device)
            total_loss += loss_fn(logits,yb).item()*len(yb); correct += (logits.argmax(1)==yb).sum().item(); count += len(yb)
    return total_loss/count, correct/count

records=[]
for epoch in range(15):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(set_to_none=True)
        loss_fn(model(xb),yb).backward(); optimizer.step()
    tr_loss,tr_acc=evaluate(train_loader); va_loss,va_acc=evaluate(valid_loader)
    records.append([epoch+1,tr_loss,va_loss,tr_acc,va_acc])
history=pd.DataFrame(records,columns=['epoch','train_loss','valid_loss','train_accuracy','valid_accuracy'])
print(history.tail(1).to_string(index=False))

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(11,4))
sns.lineplot(data=history,x='epoch',y='train_loss',label='train',ax=axes[0]); sns.lineplot(data=history,x='epoch',y='valid_loss',label='validation',ax=axes[0]); axes[0].set_title('Loss')
sns.lineplot(data=history,x='epoch',y='train_accuracy',label='train',ax=axes[1]); sns.lineplot(data=history,x='epoch',y='valid_accuracy',label='validation',ax=axes[1]); axes[1].set_title('Accuracy')
plt.tight_layout()

## 4. Freeze the decision, then inspect errors

The test set is used once after the training procedure is fixed. A confusion matrix tells us which errors occur; displaying wrong examples tells us whether they are ambiguous or reveal a data-generation problem.

In [ ]:
model.eval(); all_logits=[]
with torch.no_grad():
    for xb,_ in test_loader: all_logits.append(model(xb.to(device)).cpu())
logits=torch.cat(all_logits); predictions=logits.argmax(1).numpy(); probabilities=logits.softmax(1).numpy()
print(classification_report(y_test,predictions,target_names=['vertical','horizontal']))
sns.heatmap(confusion_matrix(y_test,predictions),annot=True,fmt='d',cmap='Blues',cbar=False,xticklabels=['vertical','horizontal'],yticklabels=['vertical','horizontal'])
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Held-out confusion matrix'); plt.show()
wrong=np.flatnonzero(predictions!=y_test)[:12]
fig,axes=plt.subplots(2,6,figsize=(12,4))
for ax,i in zip(axes.flat,wrong):
    ax.imshow(X_test[i,0],cmap='gray',vmin=0,vmax=1); ax.set_title(f'true {y_test[i]} / pred {predictions[i]}\nconfidence {probabilities[i].max():.2f}'); ax.axis('off')
plt.tight_layout()

## What a `Learner` adds

A fastai `Learner` packages the model, dataloaders, loss, optimizer, metrics, callbacks, and recorder. That is useful when the task follows standard conventions. It does not remove the need to define a valid split, choose a metric, inspect errors, or record the preprocessing. The explicit loop above is the reference against which a high-level training call should be understood.

In [ ]:
try:
    from fastai.vision.all import Learner, DataLoaders, accuracy
    print('fastai is available; an equivalent Learner can wrap the same PyTorch model and loaders.')
except ImportError:
    print('fastai is optional here; the complete experiment uses PyTorch only.')

### Limits

This dataset is deliberately clean and balanced. It does not test distribution shift, label noise, calibration, or class-dependent costs. The workflow generalizes; the numerical result does not.